# ENCM 369 — Lab 1 Solutions (Summer 2026)
Name : Rajdeep Das
UCID: 30186009


## Question 1 — Custom Logic Gate Processor

### (a) `parse_expression(expr, A, B)` — no `eval()`


In [ ]:
def AND_gate(a, b): return a & b
def OR_gate(a, b):  return a | b
def NOT_gate(a):    return ~a & 1
def XOR_gate(a, b): return a ^ b


def evaluate_no_parens(tokens):
    tokens = list(tokens)

    while 'NOT' in tokens:
        i = tokens.index('NOT')
        result = NOT_gate(int(tokens[i + 1]))
        tokens[i:i + 2] = [str(result)]

    for op, gate in [('AND', AND_gate), ('XOR', XOR_gate), ('OR', OR_gate)]:
        while op in tokens:
            i = tokens.index(op)
            result = gate(int(tokens[i - 1]), int(tokens[i + 1]))
            tokens[i - 1:i + 2] = [str(result)]

    return int(tokens[0])


def parse_expression(expr, A, B):

    expr = expr.replace('(', ' ( ').replace(')', ' ) ')
    tokens = expr.split()

    tokens = [str(A) if t == 'A' else str(B) if t == 'B' else t for t in tokens]

    while '(' in tokens:
        close_idx = tokens.index(')')
        open_idx = close_idx - 1
        while tokens[open_idx] != '(':
            open_idx -= 1
        inner_result = evaluate_no_parens(tokens[open_idx + 1:close_idx])
        tokens = tokens[:open_idx] + [str(inner_result)] + tokens[close_idx + 1:]

    return evaluate_no_parens(tokens)


# quick check
print(parse_expression("(A AND B) OR (NOT A)", A=1, B=0))

0


### (b) Evaluate `A XOR (NOT B)` for A=1, B=0
Step by step: `NOT B` = `NOT 0` = `1`, then `A XOR 1` = `1 XOR 1` = `0`.

In [ ]:
result = parse_expression("A XOR (NOT B)", A=1, B=0)
print("A XOR (NOT B), A=1, B=0 ->", result)

A XOR (NOT B), A=1, B=0 -> 0


### (c) Extend to 4-bit inputs (bitwise, e.g. A=1101, B=0110)


In [ ]:
def parse_expression_4bit(expr, A_bits, B_bits):
    assert len(A_bits) == len(B_bits)
    result_bits = []
    for i in range(len(A_bits)):
        a_bit = int(A_bits[i])
        b_bit = int(B_bits[i])
        result_bits.append(str(parse_expression(expr, a_bit, b_bit)))
    return ''.join(result_bits)

A4, B4 = "1101", "0110"
print("A       =", A4)
print("B       =", B4)
print("(A AND B) OR (NOT A) =", parse_expression_4bit("(A AND B) OR (NOT A)", A4, B4))
print("A XOR B               =", parse_expression_4bit("A XOR B", A4, B4))

A       = 1101
B       = 0110
(A AND B) OR (NOT A) = 0110
A XOR B               = 1011


### (d) Relation to hardware circuit simulation and limitations
This processor models exactly what a **combinational logic circuit** does in hardware: each `gate_*` function is the software equivalent of a physical logic gate (AND/OR/NOT/XOR)

**Two limitations compared to real digital circuits:**
1. **No timing/propagation delay.**
2. **No electrical-level behavior.** This model treats every signal as a perfect, ideal `0`/`1` with no possibility of an invalid or intermediate voltage state, so it cannot capture the electrical failure modes that occur in physical circuits.

## Question 2 — Multi-bit Adder System

### (a) Add two arbitrary-length binary strings using only bitwise logic


In [ ]:
def full_adder(a, b, cin):
    sum_bit = a ^ b ^ cin
    carry_out = (a & b) | (cin & (a ^ b))
    return sum_bit, carry_out


def add_binary(A, B):
    width = max(len(A), len(B))
    A = A.zfill(width)
    B = B.zfill(width)

    A_bits = [int(bit) for bit in A[::-1]]
    B_bits = [int(bit) for bit in B[::-1]]

    result_bits = []
    carry = 0

    for i in range(width):
        sum_bit, carry = full_adder(A_bits[i], B_bits[i], carry)
        result_bits.append(sum_bit)

    if carry:
        result_bits.append(carry)

    result_bits_reversed = result_bits[::-1]
    return "".join(str(bit) for bit in result_bits_reversed)

### (b) Demonstrate on the given 16-bit values

In [ ]:
A = "1101001101010101"
B = "1011101110001110"

sum_str = add_binary(A, B)
print("A           =", A)
print("B           =", B)
print("Binary sum  =", sum_str)
print("Decimal sum =", int(sum_str, 2))


A           = 1101001101010101
B           = 1011101110001110
Binary sum  = 11000111011100011
Decimal sum = 102115


### (c) Detect overflow in 16-bit two's-complement arithmetic


In [ ]:
def add_binary_overflow(A, B):
    A_bits = [int(b) for b in A.zfill(16)[::-1]]
    B_bits = [int(b) for b in B.zfill(16)[::-1]]

    result_bits, carry = [], 0
    for i in range(16):
        if i == 15:
            carry_into_sign = carry
        s, carry = full_adder(A_bits[i], B_bits[i], carry)
        result_bits.append(s)

    overflow = (carry_into_sign != carry)
    sum_str = "".join(str(b) for b in result_bits[::-1])
    return sum_str, overflow

A = "0111111111111111"
B = "0000000000000001"
sum_str, overflow = add_binary_overflow(A, B)
print("A =", A, "(+32767)")
print("B =", B, "(+1)")
print("16-bit sum    :", sum_str)
print("Overflow flag :", overflow)

A = 0111111111111111 (+32767)
B = 0000000000000001 (+1)
16-bit sum    : 1000000000000000
Overflow flag : True


### (d) Why hardware ALUs must handle carry propagation, and the timing impact
Hardware ALUs must propagate carries because each bit's sum depends on the carry generated by all less-significant bits.
This makes the adder's delay grow linearly with bit-width, so for wide (32/64-bit) operands the carry path becomes the critical path that caps the CPU's clock speed

## Question 3 — ASCII Memory Map & Visualization

### (a) Store a string as 8-bit ASCII codes in a simulated memory dictionary


In [ ]:
def store_message_in_memory(message, base_address=0x1000):
    memory = {}
    addr = base_address
    for ch in message:
        ascii_code = ord(ch)
        binary_code = format(ascii_code, '08b')
        memory[addr] = binary_code
        addr += 1
    return memory

message = "Hi!"
memory = store_message_in_memory(message)
for addr, val in memory.items():
    print(hex(addr), val)

0x1000 01001000
0x1001 01101001
0x1002 00100001


### (b) Print a hex memory dump (debugger-style)

In [ ]:
def hex_memory_dump(memory):
    print(f"{'ADDRESS':<10}{'HEX':<6}{'BINARY':<12}{'CHAR'}")
    for addr in sorted(memory.keys()):
        binary_val = memory[addr]
        byte_val = int(binary_val, 2)
        hex_val = format(byte_val, '02X')
        char_repr = chr(byte_val) if 32 <= byte_val <= 126 else '.'
        print(f"0x{addr:04X}    {hex_val:<6}{binary_val:<12}{char_repr}")

hex_memory_dump(memory)

ADDRESS   HEX   BINARY      CHAR
0x1000    48    01001000    H
0x1001    69    01101001    i
0x1002    21    00100001    !


### (c) Handle non-printable characters and flag them in the dump

In [ ]:
def hex_memory_dump_extended(memory):
    print(f"{'ADDRESS':<10}{'HEX':<6}{'BINARY':<12}{'CHAR':<6}{'PRINTABLE?'}")
    for addr in sorted(memory.keys()):
        binary_val = memory[addr]
        byte_val = int(binary_val, 2)
        hex_val = format(byte_val, '02X')
        is_printable = 32 <= byte_val <= 126
        char_repr = chr(byte_val) if is_printable else '.'
        flag = "yes" if is_printable else "NON-PRINTABLE"
        print(f"0x{addr:04X}    {hex_val:<6}{binary_val:<12}{char_repr:<6}{flag}")

message2 = "Hi\n!\t"
memory2 = store_message_in_memory(message2)
hex_memory_dump_extended(memory2)

ADDRESS   HEX   BINARY      CHAR  PRINTABLE?
0x1000    48    01001000    H     yes
0x1001    69    01101001    i     yes
0x1002    0A    00001010    .     NON-PRINTABLE
0x1003    21    00100001    !     yes
0x1004    09    00001001    .     NON-PRINTABLE


### (d) Why text encodings matter in low-level systems
ASCII and Unicode define the mapping between raw bytes in memory and the characters a program (or a human) interprets them as. At the hardware/software boundary there is no inherent difference between text and any other binary data as in a byte is just a byte therefore the encoding is the only thing that gives those bytes meaning.

## Question 4 — CPU Microprogram Simulator

### (a) Microinstruction set as a Python data structure


In [ ]:
microprogram = [
    {"opcode": "LOAD",  "src": "[100]", "dest": "R1"},
    {"opcode": "ADD",   "src": 3,       "dest": "R1"},
    {"opcode": "MOVE",  "src": "R1",    "dest": "R2"},
    {"opcode": "SUB",   "src": 1,       "dest": "R2"},
    {"opcode": "STORE", "src": "R2",    "dest": "[200]"}
]

for instr in microprogram:
    print(instr)

{'opcode': 'LOAD', 'src': '[100]', 'dest': 'R1'}
{'opcode': 'ADD', 'src': 3, 'dest': 'R1'}
{'opcode': 'MOVE', 'src': 'R1', 'dest': 'R2'}
{'opcode': 'SUB', 'src': 1, 'dest': 'R2'}
{'opcode': 'STORE', 'src': 'R2', 'dest': '[200]'}


### (b)/(c) Execute the microprogram and track the program counter (PC)

In [ ]:
def run_microprogram(microprogram, memory, registers):
    pc = 0
    for instr in microprogram:
        opcode, src, dest = instr["opcode"], instr["src"], instr["dest"]

        if opcode == "LOAD":
            addr = int(src.strip("[]"))
            registers[dest] = memory[addr]
        elif opcode == "ADD":
            value = registers[src] if (isinstance(src, str) and src in registers) else src
            registers[dest] = registers[dest] + value
        elif opcode == "SUB":
            value = registers[src] if (isinstance(src, str) and src in registers) else src
            registers[dest] = registers[dest] - value
        elif opcode == "MOVE":
            registers[dest] = registers[src]
        elif opcode == "STORE":
            addr = int(dest.strip("[]"))
            memory[addr] = registers[src]
        else:
            raise ValueError(f"Unknown opcode: {opcode}")

        pc += 1
        print(f"PC={pc}: {opcode:5s} {str(src):6s} {dest:5s} -> registers={registers}  memory={memory}")

    return memory, registers

Memory = {100: 7, 200: 4}
Registers = {"R1": 0, "R2": 0}

final_memory, final_registers = run_microprogram(microprogram, Memory, Registers)
print("\nFinal memory   :", final_memory)
print("Final registers:", final_registers)

PC=1: LOAD  [100]  R1    -> registers={'R1': 7, 'R2': 0}  memory={100: 7, 200: 4}
PC=2: ADD   3      R1    -> registers={'R1': 10, 'R2': 0}  memory={100: 7, 200: 4}
PC=3: MOVE  R1     R2    -> registers={'R1': 10, 'R2': 10}  memory={100: 7, 200: 4}
PC=4: SUB   1      R2    -> registers={'R1': 10, 'R2': 9}  memory={100: 7, 200: 4}
PC=5: STORE R2     [200] -> registers={'R1': 10, 'R2': 9}  memory={100: 7, 200: 9}

Final memory   : {100: 7, 200: 9}
Final registers: {'R1': 10, 'R2': 9}


### (d) Microprogrammed vs. hardwired control units
A **hardwired control unit** implements the CPU's control logic directly as fixed combinational (and sequential) circuit with a network of gates and a finite-state machine wired to assert exactly the control signals each instruction needs.
A **microprogrammed control unit**, in contrast, stores each machine instruction's behavior as a sequence of **microinstructions** in a control-store memory.

Microprogramming is used in modern CPU design because it turns control-logic design into a software-like problem so instructions can be added, extended, or patched by rewriting microcode rather than redesigning silicon.

## Question 5 — Integer Overflow & Security

### (a) Simulate 8-bit signed integer arithmetic


In [ ]:
def add_8bit_signed(A, B):
    def to_twos_complement(n, bits=8):
        if n < 0:
            n = (1 << bits) + n
        return n & ((1 << bits) - 1)

    def from_twos_complement(n, bits=8):
        if n & (1 << (bits - 1)):
            n -= (1 << bits)
        return n

    a_bits = to_twos_complement(A)
    b_bits = to_twos_complement(B)

    raw_sum = a_bits + b_bits
    result_bits = raw_sum & 0xFF

    signed_result = from_twos_complement(result_bits)

    a_sign = (a_bits >> 7) & 1
    b_sign = (b_bits >> 7) & 1
    r_sign = (result_bits >> 7) & 1
    overflow = (a_sign == b_sign) and (r_sign != a_sign)

    return signed_result, overflow, format(a_bits, '08b'), format(b_bits, '08b'), format(result_bits, '08b')

### (b) Test with A=127, B=1 — show the binary steps

In [ ]:
A, B = 127, 1
signed_result, overflow, a_bin, b_bin, r_bin = add_8bit_signed(A, B)

print(f"A = {A:>4}  ->  {a_bin}")
print(f"B = {B:>4}  ->  {b_bin}")
print(f"{'-'*20}")
print(f"raw sum bits      ->  {r_bin}")
print(f"signed result     =  {signed_result}")
print(f"overflow occurred =  {overflow}")

A =  127  ->  01111111
B =    1  ->  00000001
--------------------
raw sum bits      ->  10000000
signed result     =  -128
overflow occurred =  True


### (c) Example of overflow becoming a security vulnerability
A shopping cart stores quantity as an 8-bit unsigned integer and computes total = price × quantity. An attacker sets quantity = 256, which wraps to 0, so the total cost becomes $0

### (d) How languages/compilers mitigate integer overflow
Python uses dynamic sizing to allow it to grow till system memory permits. C/C++ performs 'wrap around' to prevent a program crash.